# Dataset Perturbation Preparation

This notebook loads the prepared fake-news datasets, applies the proposed perturbation pipeline, and saves the perturbed outputs to Google Drive.

## Workflow
1. Set up imports and mount Google Drive.
2. Load the datasets.
3. Define the perturbation steps.
4. Apply perturbations to each loaded dataset.
5. Save the perturbed datasets to Google Drive.
6. Verify the saved outputs.

# Section 1: Set Up Imports and Mount Google Drive

This section installs the text augmentation dependency if needed, mounts Google Drive, and defines the input and output locations.

In [ ]:
import os
import sys
import subprocess
from pathlib import Path

import pandas as pd
from google.colab import drive

try:
    import nlpaug.augmenter.word as naw
except ImportError:
    print("Installing nlpaug...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "nlpaug"])
    import nlpaug.augmenter.word as naw

# Mount Google Drive so the notebook can read and write files.
drive.mount('/content/drive')

BASE_DATA_DIR = Path('/content/drive/MyDrive/datasets')
OUTPUT_DIR = BASE_DATA_DIR / 'perturbed_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Source datasets to load.
DATASET_SPECS = {
    'WELFake': BASE_DATA_DIR / 'WELFake_processed.csv',
    'FakeNewsNet': BASE_DATA_DIR / 'FakeNewsNet_processed.csv',
    'Fake_News_Detection': BASE_DATA_DIR / 'Fake_News_Detection_processed.csv',
    'ISOT': BASE_DATA_DIR / 'ISOT_processed.csv',
    'Fake_News_Classification': BASE_DATA_DIR / 'Fake_News_Classification_processed.csv',
}

print(f'Output directory: {OUTPUT_DIR}')

# Section 2: Load the Datasets

Load the prepared CSV datasets into memory and make sure the expected text and label columns are present.

In [ ]:
def normalize_dataset_frame(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy().dropna().reset_index(drop=True)

    if 'label' not in df.columns and 'labels' in df.columns:
        df = df.rename(columns={'labels': 'label'})

    if 'combined_text' not in df.columns:
        possible_pairs = [
            ('title', 'text'),
            ('title', 'content'),
            ('headline', 'text'),
            ('title', 'body'),
        ]
        for first_col, second_col in possible_pairs:
            if first_col in df.columns and second_col in df.columns:
                df['combined_text'] = (
                    df[first_col].fillna('').astype(str).str.strip() + ' ' +
                    df[second_col].fillna('').astype(str).str.strip()
                ).str.strip()
                break

    if 'combined_text' not in df.columns:
        raise ValueError('Dataset must contain a combined_text column or a recognizable text pair.')
    if 'label' not in df.columns:
        raise ValueError('Dataset must contain a label column.')

    return df

loaded_datasets = {}
for name, path in DATASET_SPECS.items():
    if not path.exists():
        raise FileNotFoundError(f'Missing dataset: {path}')
    frame = pd.read_csv(path)
    frame = normalize_dataset_frame(frame)
    loaded_datasets[name] = frame
    print(f"Loaded {name}: {len(frame):,} rows | columns: {list(frame.columns)[:8]}")

# Section 3: Define the Proposed Model Perturbation Steps

The perturbation pipeline performs synonym replacement first, then random swapping, then random deletion.

In [ ]:
class TextPerturber:
    def __init__(self, delete_prob=0.075, swap_prob=0.075, substitute_prob=0.15):
        self.delete_prob = delete_prob
        self.swap_prob = swap_prob
        self.substitute_prob = substitute_prob

        self.aug_sub = naw.SynonymAug(aug_src='wordnet', aug_p=self.substitute_prob)
        self.aug_swap = naw.RandomWordAug(action='swap', aug_p=self.swap_prob)
        self.aug_del = naw.RandomWordAug(action='delete', aug_p=self.delete_prob)

    def _first_value(self, augmented_text):
        if isinstance(augmented_text, list):
            return augmented_text[0]
        return augmented_text

    def perturb(self, text: str) -> str:
        augmented_text = self._first_value(self.aug_sub.augment(str(text)))
        augmented_text = self._first_value(self.aug_swap.augment(augmented_text))
        augmented_text = self._first_value(self.aug_del.augment(augmented_text))
        return augmented_text

# Section 4: Apply Perturbations to the Loaded Datasets

Run the perturbation pipeline over each loaded dataset and keep the perturbed data in memory before saving.

In [ ]:
perturber = TextPerturber()
perturbed_datasets = {}

for name, frame in loaded_datasets.items():
    perturbed_frame = frame.copy()
    perturbed_frame['combined_text_perturbed'] = [perturber.perturb(text) for text in perturbed_frame['combined_text'].astype(str).tolist()]
    perturbed_datasets[name] = perturbed_frame
    print(f'Perturbed {name}: {len(perturbed_frame):,} rows')

# Section 5: Save the Perturbed Datasets to Google Drive

Save each perturbed dataset to the output directory in Google Drive.

In [ ]:
saved_paths = {}
for name, frame in perturbed_datasets.items():
    output_path = OUTPUT_DIR / f'{name}_perturbed.csv'
    frame.to_csv(output_path, index=False)
    saved_paths[name] = output_path
    print(f'Saved {name} to {output_path}')

# Section 6: Verify Saved Outputs

Reload the saved files from Google Drive and confirm the output column is present.

In [ ]:
for name, output_path in saved_paths.items():
    reloaded = pd.read_csv(output_path)
    if 'combined_text_perturbed' not in reloaded.columns:
        raise ValueError(f'Missing perturbed column in {output_path}')
    print(f'Verified {name}: {len(reloaded):,} rows | columns: {list(reloaded.columns)[:8]}')

print('All perturbed datasets were saved and reloaded successfully.')